In [1]:
import sympy as sym
import numpy as np
import dill

sym.init_printing(use_unicode=True)

print(np.__version__)

2.5.2


# Intro

## Preliminaries

\begin{align*}
q\{\phi \} &\triangleq \text{Exp}(\phi) = \begin{bmatrix} \cos(\frac{||\phi||}{2}) \\ \frac{\phi}{||\phi||} \sin(\frac{||\phi||}{2}) \end{bmatrix} \approx \begin{bmatrix} 1 \\ \frac{\phi}{2} \end{bmatrix} \text{ for small angles.} \\
\\
[\omega]_\times &\triangleq \begin{bmatrix}
0 & -\omega_z & \omega_y \\
\omega_z & 0 & -\omega_x \\
-\omega_y & \omega_x & 0
\end{bmatrix}
\end{align*}

In [2]:
def quat_from_axis_angle(Phi, small_angle=True):
    if small_angle:
        return sym.Matrix([1, *(0.5 * Phi)])
    
    norm = sym.sqrt(Phi[0]**2 + Phi[1]**2 + Phi[2]**2)
    return sym.Matrix([
        sym.cos(0.5 * norm),
        *(Phi * (1 / norm) * sym.sin(0.5 * norm))
    ])

def skew_symmetric_matrix(omega):
    omega_x, omega_y, omega_z = omega[0], omega[1], omega[2]
    return sym.Matrix([
        [0, -omega_z, omega_y],
        [omega_z, 0, -omega_x],
        [-omega_y, omega_x, 0]
    ])

## Helper Functions

In [3]:
# rotation matrix from quaternion
def R(q):
    qw, qx, qy, qz = q[0], q[1], q[2], q[3]
    return sym.Matrix([
        [1 - 2*(qy**2 + qz**2), 2*(qx*qy - qz*qw),     2*(qx*qz + qy*qw)],
        [2*(qx*qy + qz*qw),     1 - 2*(qx**2 + qz**2), 2*(qy*qz - qx*qw)],
        [2*(qx*qz - qy*qw),     2*(qy*qz + qx*qw),     1 - 2*(qx**2 + qy**2)]
    ])

def quat_inv(q):
    return sym.Matrix([q[0], -q[1], -q[2], -q[3]])
    
def quat_norm(q):
    return q / sym.sqrt(q[0]**2 + q[1]**2 + q[2]**2 + q[3]**2)


def left_quat_matrix(q):
    qw, qx, qy, qz = q[0], q[1], q[2], q[3]
    
                            #w
                            #x
                            #y
                            #z
    return sym.Matrix([
        [qw, -qx, -qy, -qz], 
        [qx,  qw, -qz,  qy], 
        [qy,  qz,  qw, -qx], 
        [qz, -qy,  qx,  qw]  
    ])

# def right_quat_matrix(q):
#     qw, qx, qy, qz = q[0], q[1], q[2], q[3]
#     return sym.Matrix([
#         [qw, -qx, -qy, -qz], 
#         [qx,  qw,  qz, -qy], 
#         [qy, -qz,  qw,  qx], 
#         [qz,  qy, -qx,  qw]  
#     ])
# 
# def rotate_point(point, q):
#     return sym.Matrix((
#         left_quat_matrix(q) * right_quat_matrix(quat_inv(q)) * sym.Matrix([0, *point])
#     )[1:4])

## Additional Variables

For each cycle, we calculate $\Delta t$ in code and we assume a fixed gravity acceleration force. For the offset between IMU and camera, we define translation and rotation.

\begin{align*}
\Delta p_{I \to c} &= \begin{bmatrix}
0 & -0.0013 & - 0.00662
\end{bmatrix}^\top \\
\Delta q_{I \to c} &= \begin{bmatrix}
\displaystyle \cos\left(-\frac{102°}{2}\right) & \displaystyle \sin\left(-\frac{102°}{2}\right) & 0 & 0
\end{bmatrix}^\top \\
\end{align*}

In [4]:
dt = sym.Symbol(r'\Delta t', real=True, positive=True)
gravity = sym.Symbol('g', real=True, positive=True)

# [0, -0.0013, -0.00662]
IMU_to_cam_translation = sym.Matrix(sym.symbols(r'\Delta{p}_x \Delta{p}_y \Delta{p}_z', real=True))

# concise version
# Phi = 102 * (constants.pi / 180) 
# angle = sym.Symbol("\\phi")
# IMU_to_cam_rotation = sym.Matrix([sym.cos(-angle / 2), sym.sin(-angle / 2), 0, 0])

# simplified version for observability calculations
c_phi_2, s_phi_2 = sym.symbols(r'c_{\phi/2} s_{\phi/2}', real=True)
IMU_to_cam_rotation = sym.Matrix([c_phi_2, s_phi_2, 0, 0])

dt, gravity, IMU_to_cam_translation, IMU_to_cam_rotation

⎛                            ⎡c_{\phi/2}⎤⎞
⎜             ⎡\Delta{p}ₓ ⎤  ⎢          ⎥⎟
⎜             ⎢           ⎥  ⎢s_{\phi/2}⎥⎟
⎜\Delta t, g, ⎢\Delta{p}_y⎥, ⎢          ⎥⎟
⎜             ⎢           ⎥  ⎢    0     ⎥⎟
⎜             ⎣\Delta{p}_z⎦  ⎢          ⎥⎟
⎝                            ⎣    0     ⎦⎠

---
# Input

\begin{align*}
&\text{Accelerometer input: } &\tilde a && &\text{noise: } &\tilde a_n \\
&\text{Gyroscope input: } &\tilde \omega && &\text{noise: } &\tilde \omega_n\\
\end{align*}

In [5]:
a_tilde = sym.Matrix(sym.symbols(r"\tilde{a}_x \tilde{a}_y \tilde{a}_z", real=True))
omega_tilde = sym.Matrix(sym.symbols(r"\tilde{\omega}_x \tilde{\omega}_y \tilde{\omega}_z", real=True))
u_tilde = sym.Matrix.vstack(a_tilde, omega_tilde)

a_tilde_n = sym.Matrix(sym.symbols(r"\tilde{a}_{n\,x} \tilde{a}_{n\,y} \tilde{a}_{n\,z}", real=True))
omega_tilde_n = sym.Matrix(sym.symbols(r"\tilde{\omega}_{n\,x} \tilde{\omega}_{n\,y} \tilde{\omega}_{n\,z}", real=True))
u_tilde_n = sym.Matrix.vstack(a_tilde_n, omega_tilde_n)


u_tilde.T, u_tilde_n.T

([\tilde{a}ₓ  \tilde{a}_y  \tilde{a}_z  \tilde{\omega}ₓ  \tilde{\omega}_y  \ti ↪

↪ lde{\omega}_z], [\tilde{a}_{n,x}  \tilde{a}_{n,y}  \tilde{a}_{n,z}  \tilde{\ ↪

↪ omega}_{n,x}  \tilde{\omega}_{n,y}  \tilde{\omega}_{n,z}])

---
# Nominal State

## IMU

\begin{align*}
x_\text{IMU} = \begin{bmatrix} p & v & q & a_b & \omega_b \end{bmatrix}^\top \qquad \qquad x \in SO(3) \times \left(\mathbb{R}^3\right)^4
\end{align*}

In [6]:
p = sym.Matrix(sym.symbols(r"p_x p_y p_z", real=True))                                     # position
v = sym.Matrix(sym.symbols(r"v_x v_y v_z", real=True))                                     # velocity
q = sym.Matrix(sym.symbols(r"q_w q_x q_y q_z", real=True))                                 # orientation quaternion
a_b = sym.Matrix(sym.symbols(r"a_{b\,x} a_{b\,y} a_{b\,z}", real=True))                    # accelerometer bias
omega_b = sym.Matrix(sym.symbols(r"\omega_{b\,x} \omega_{b\,y} \omega_{b\,z}", real=True)) # gyroscope bias

x_IMU = sym.Matrix.vstack(p, v, q, a_b, omega_b)
x_IMU.T

[pₓ  p_y  p_z  vₓ  v_y  v_z  q_w  qₓ  q_y  q_z  a_{b,x}  a_{b,y}  a_{b,z}  \om ↪

↪ ega_{b,x}  \omega_{b,y}  \omega_{b,z}]

### Kinematics

\begin{align*}
p &\leftarrow p + v\Delta t + \frac{1}{2}( \mathbf{R}^{\{q\}} (\tilde a - a_b) + g)\Delta t^2 \qquad & a_b &\leftarrow a_b \\
\text{Discrete Case} \qquad \qquad v &\leftarrow v + (\mathbf{R}^{\{q\}}(\tilde a - a_b) + g)\Delta t & \omega_b &\leftarrow \omega_b \\
q &\leftarrow q \otimes q\{(\tilde \omega - \omega_b) \Delta t\} \\
\end{align*}

In [7]:
f_p = p + v * dt + 0.5 * (R(q) * (a_tilde - a_b) + sym.Matrix([0, 0, gravity])) * dt**2
f_v = v + (R(q) * (a_tilde - a_b) + sym.Matrix([0, 0, gravity])) * dt
f_q = left_quat_matrix(q) * quat_from_axis_angle((omega_tilde - omega_b) * dt)
f_a_b = a_b
f_omega_b = omega_b

f_nominal_state_IMU = sym.Matrix.vstack(f_p, f_v, f_q, f_a_b, f_omega_b)

f_nominal_state_IMU

⎡            2 ⎛                           ⎛       2        2    ⎞             ↪
⎢    \Delta t ⋅⎝0.5⋅(\tilde{a}ₓ - a_{b,x})⋅⎝- 2⋅q_y  - 2⋅q_z  + 1⎠ + 0.5⋅(\til ↪
⎢                                                                              ↪
⎢            2 ⎛                                                               ↪
⎢    \Delta t ⋅⎝0.5⋅(\tilde{a}ₓ - a_{b,x})⋅(2⋅q_w⋅q_z + 2⋅qₓ⋅q_y) + 0.5⋅(\tild ↪
⎢                                                                              ↪
⎢        2 ⎛                                                                   ↪
⎢\Delta t ⋅⎝0.5⋅g + 0.5⋅(\tilde{a}ₓ - a_{b,x})⋅(-2⋅q_w⋅q_y + 2⋅qₓ⋅q_z) + 0.5⋅( ↪
⎢                                                                              ↪
⎢                           ⎛                       ⎛       2        2    ⎞    ↪
⎢                  \Delta t⋅⎝(\tilde{a}ₓ - a_{b,x})⋅⎝- 2⋅q_y  - 2⋅q_z  + 1⎠ +  ↪
⎢                                                                              ↪
⎢                           

## Speakers

\begin{align*}
x_s = \begin{bmatrix} p_i & v_i & q_i \end{bmatrix}^\top \qquad \qquad x_s \in SO(3) \times \left(\mathbb{R}^3\right)^2
\end{align*}

In [8]:
p_i = sym.Matrix(sym.symbols(r"p_{i\,x} p_{i\,y} p_{i\,z}", real=True))
v_i = sym.Matrix(sym.symbols(r"v_{i\,x} v_{i\,y} v_{i\,z}", real=True))
q_i = sym.Matrix(sym.symbols(r"q_{i\,w} q_{i\,x} q_{i\,y} q_{i\,z}", real=True))

x_speaker = sym.Matrix.vstack(p_i, v_i, q_i)
x_speaker.T

[p_{i,x}  p_{i,y}  p_{i,z}  v_{i,x}  v_{i,y}  v_{i,z}  q_{i,w}  q_{i,x}  q_{i, ↪

↪ y}  q_{i,z}]

### Kinematics

\begin{align*}
p_i &\leftarrow p_i + v_i \Delta t \\
\text{Discrete Case} \qquad \qquad v_i &\leftarrow v_i \\
q_i &\leftarrow q_i \\
\end{align*}

In [9]:
f_p_i = p_i + v_i * dt
f_v_i = v_i
f_q_i = q_i

f_nominal_state_speaker = sym.Matrix.vstack(f_p_i, f_v_i, f_q_i)
# f_nominal_state_speaker

## Combined State

\begin{align*}
x = \begin{bmatrix} p & v & q & a_b & \omega_b & p_i & v_i & q_i & p_{i + 1} & v_{i + 1} & q_{i + 1} & \cdots \end{bmatrix}^\top
\end{align*}

In [10]:
x = sym.Matrix.vstack(x_IMU, x_speaker)
f_nominal_state = sym.Matrix.vstack(f_nominal_state_IMU, f_nominal_state_speaker)

x.T

[pₓ  p_y  p_z  vₓ  v_y  v_z  q_w  qₓ  q_y  q_z  a_{b,x}  a_{b,y}  a_{b,z}  \om ↪

↪ ega_{b,x}  \omega_{b,y}  \omega_{b,z}  p_{i,x}  p_{i,y}  p_{i,z}  v_{i,x}  v ↪

↪ _{i,y}  v_{i,z}  q_{i,w}  q_{i,x}  q_{i,y}  q_{i,z}]

---
# Error State

## IMU

\begin{align*}
\delta x_\text{IMU} = \begin{bmatrix} \delta p & \delta v & \delta \theta & \delta a_b & \delta \omega_b \end{bmatrix}^\top \qquad \qquad x \in \left(\mathbb{R}^3\right)^5
\end{align*}

In [11]:
delta_p = sym.Matrix(sym.symbols(r"\delta{p}_x \delta{p}_y \delta{p}_z", real=True))
delta_v = sym.Matrix(sym.symbols(r"\delta{v}_x \delta{v}_y \delta{v}_z", real=True))
delta_theta = sym.Matrix(sym.symbols(r"\delta{\theta}_x \delta{\theta}_y \delta{\theta}_z", real=True))
delta_a_b = sym.Matrix(sym.symbols(r"\delta{a}_{b\,x} \delta{a}_{b\,y} \delta{a}_{b\,z}", real=True))
delta_omega_b = sym.Matrix(sym.symbols(r"\delta{\omega}_{b\,x} \delta{\omega}_{b\,y} \delta{\omega}_{b\,z}", real=True))

delta_x_IMU = sym.Matrix.vstack(delta_p, delta_v, delta_theta, delta_a_b, delta_omega_b)

# IMU Process Noise
delta_v_n = sym.Matrix(sym.symbols(r"\delta{v}_{n\,x} \delta{v}_{n\,y} \delta{v}_{n\,z}", real=True))
delta_theta_n = sym.Matrix(sym.symbols(r"\delta{\theta}_{n\,x} \delta{\theta}_{n\,y} \delta{\theta}_{n\,z}", real=True))
delta_a_n = sym.Matrix(sym.symbols(r"\delta{a}_{n\,x} \delta{a}_{n\,y} \delta{a}_{n\,z}", real=True))
delta_omega_n = sym.Matrix(sym.symbols(r"\delta{\omega}_{n\,x} \delta{\omega}_{n\,y} \delta{\omega}_{n\,z}", real=True))

delta_x_n_IMU = sym.Matrix.vstack(delta_v_n, delta_theta_n, delta_a_n, delta_omega_n)

delta_x_IMU.T, delta_x_n_IMU.T

([\delta{p}ₓ  \delta{p}_y  \delta{p}_z  \delta{v}ₓ  \delta{v}_y  \delta{v}_z   ↪

↪ \delta{\theta}ₓ  \delta{\theta}_y  \delta{\theta}_z  \delta{a}_{b,x}  \delta ↪

↪ {a}_{b,y}  \delta{a}_{b,z}  \delta{\omega}_{b,x}  \delta{\omega}_{b,y}  \del ↪

↪ ta{\omega}_{b,z}], [\delta{v}_{n,x}  \delta{v}_{n,y}  \delta{v}_{n,z}  \delt ↪

↪ a{\theta}_{n,x}  \delta{\theta}_{n,y}  \delta{\theta}_{n,z}  \delta{a}_{n,x} ↪

↪   \delta{a}_{n,y}  \delta{a}_{n,z}  \delta{\omega}_{n,x}  \delta{\omega}_{n, ↪

↪ y}  \delta{\omega}_{n,z}])

### Kinematics

\begin{align*}
\delta p &\leftarrow \delta p + \delta v\Delta t & \delta a_b &\leftarrow \delta a_b + \delta a_n \\
\text{Discrete Case} \qquad \qquad \delta v &\leftarrow \delta v + (-\mathbf{R}^{\{q\}}[\tilde a - a_b]_\times \delta \theta - \mathbf{R}^{\{q\}} \delta a_b)\Delta t + \delta v_{n} \qquad & \delta \omega_b &\leftarrow \delta \omega_b + \delta \omega_n \\
\delta \theta &\leftarrow (\mathbf{R}^{\{(\tilde \omega - \omega_b) \Delta t\}})^\top \delta \theta - \delta \omega_b \Delta t + \delta \theta_{n}
\end{align*}

In [12]:
f_delta_p = delta_p + delta_v * dt
f_delta_v = delta_v + (- R(q) * skew_symmetric_matrix(a_tilde - a_b) * delta_theta \
    - R(q) * delta_a_b) * dt + delta_v_n
f_delta_theta = R(quat_from_axis_angle((omega_tilde - omega_b) * dt)).T * delta_theta \
    - delta_omega_b * dt + delta_theta_n
f_delta_a_b = delta_a_b + delta_a_n
f_delta_omega_b = delta_omega_b + delta_omega_n

f_error_state_IMU = sym.Matrix.vstack(f_delta_p, f_delta_v, f_delta_theta, f_delta_a_b, f_delta_omega_b)

# print(sym.latex(f_error_state_transition))
f_error_state_IMU

⎡                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢         ⎛                                                                    ↪
⎢\Delta t⋅⎝\delta{\theta}ₓ⋅((-\tilde{a}_y + a_{b,y})⋅(-2⋅q_w⋅q_y - 2⋅qₓ⋅q_z) + ↪
⎢                                                                              ↪
⎢          ⎛                ⎛                                                  ↪
⎢ \Delta t⋅⎝\delta{\theta}ₓ⋅⎝(-\tilde{a}_y + a_{b,y})⋅(2⋅q_w⋅qₓ - 2⋅q_y⋅q_z) + ↪
⎢                                                                              ↪
⎢          ⎛                

## Speakers

\begin{align*}
\delta x_s = \begin{bmatrix} \delta p_i & \delta v_i & \delta \theta_i \end{bmatrix}^\top \qquad \qquad x_s \in \left(\mathbb{R}^3\right)^3
\end{align*}

In [13]:
delta_p_i = sym.Matrix(sym.symbols(r"\delta{p}_{i\,x} \delta{p}_{i\,y} \delta{p}_{i\,z}", real=True))
delta_v_i = sym.Matrix(sym.symbols(r"\delta{v}_{i\,x} \delta{v}_{i\,y} \delta{v}_{i\,z}", real=True))
delta_theta_i = sym.Matrix(sym.symbols(r"\delta{\theta}_{i\,x} \delta{\theta}_{i\,y} \delta{\theta}_{i\,z}", real=True))

delta_x_speaker = sym.Matrix.vstack(delta_p_i, delta_v_i, delta_theta_i)

# Speaker Process Noise
delta_p_i_n = sym.Matrix(sym.symbols(r"\delta{p}_{in\,x} \delta{p}_{in\,y} \delta{p}_{in\,z}", real=True))
delta_v_i_n = sym.Matrix(sym.symbols(r"\delta{v}_{in\,x} \delta{v}_{in\,y} \delta{v}_{in\,z}", real=True))
delta_theta_i_n = sym.Matrix(sym.symbols(r"\delta{\theta}_{in\,x} \delta{\theta}_{in\,y} \delta{\theta}_{in\,z}", real=True))

delta_x_n_speaker = sym.Matrix.vstack(delta_p_i_n, delta_v_i_n, delta_theta_i_n)

delta_x_speaker.T, delta_x_n_speaker.T


([\delta{p}_{i,x}  \delta{p}_{i,y}  \delta{p}_{i,z}  \delta{v}_{i,x}  \delta{v ↪

↪ }_{i,y}  \delta{v}_{i,z}  \delta{\theta}_{i,x}  \delta{\theta}_{i,y}  \delta ↪

↪ {\theta}_{i,z}], [\delta{p}_{in,x}  \delta{p}_{in,y}  \delta{p}_{in,z}  \del ↪

↪ ta{v}_{in,x}  \delta{v}_{in,y}  \delta{v}_{in,z}  \delta{\theta}_{in,x}  \de ↪

↪ lta{\theta}_{in,y}  \delta{\theta}_{in,z}])

### Kinematics

\begin{align*}
\delta p_i &\leftarrow \delta p_i + \delta v_i \Delta t + \delta p_{in} \\
\text{Discrete Case} \qquad \qquad \delta v_i &\leftarrow \delta v_i + \delta v_{in} \\
\delta \theta_i &\leftarrow \delta \theta_i + \delta \theta_{in}\\
\end{align*}

In [14]:
f_delta_p_i = delta_p_i + delta_v_i * dt + delta_p_i_n
f_delta_v_i = delta_v_i + delta_v_i_n
f_delta_theta_i = delta_theta_i + delta_theta_i_n

f_error_state_speaker = sym.Matrix.vstack(f_delta_p_i, f_delta_v_i, f_delta_theta_i)
f_error_state_speaker

⎡\Delta t⋅\delta{v}_{i,x} + \delta{p}_{i,x} + \delta{p}_{in,x}⎤
⎢                                                             ⎥
⎢\Delta t⋅\delta{v}_{i,y} + \delta{p}_{i,y} + \delta{p}_{in,y}⎥
⎢                                                             ⎥
⎢\Delta t⋅\delta{v}_{i,z} + \delta{p}_{i,z} + \delta{p}_{in,z}⎥
⎢                                                             ⎥
⎢             \delta{v}_{i,x} + \delta{v}_{in,x}              ⎥
⎢                                                             ⎥
⎢             \delta{v}_{i,y} + \delta{v}_{in,y}              ⎥
⎢                                                             ⎥
⎢             \delta{v}_{i,z} + \delta{v}_{in,z}              ⎥
⎢                                                             ⎥
⎢        \delta{\theta}_{i,x} + \delta{\theta}_{in,x}         ⎥
⎢                                                             ⎥
⎢        \delta{\theta}_{i,y} + \delta{\theta}_{in,y}         ⎥
⎢                                       

## Combined State

\begin{align*}
\delta x = \begin{bmatrix} \delta p & \delta v & \delta \theta & \delta a_b & \delta \omega_b & \delta p_i & \delta v_i & \delta \theta_i & \delta p_{i + 1} & \delta v_{i + 1} & \delta \theta_{i + 1} & \cdots \end{bmatrix}^\top \\
\end{align*}

In [15]:
delta_x = sym.Matrix.vstack(delta_x_IMU, delta_x_speaker)
f_error_state = sym.Matrix.vstack(f_error_state_IMU, f_error_state_speaker)

delta_x_n = sym.Matrix.vstack(delta_x_n_IMU, delta_x_n_speaker)

delta_x.T

[\delta{p}ₓ  \delta{p}_y  \delta{p}_z  \delta{v}ₓ  \delta{v}_y  \delta{v}_z  \ ↪

↪ delta{\theta}ₓ  \delta{\theta}_y  \delta{\theta}_z  \delta{a}_{b,x}  \delta{ ↪

↪ a}_{b,y}  \delta{a}_{b,z}  \delta{\omega}_{b,x}  \delta{\omega}_{b,y}  \delt ↪

↪ a{\omega}_{b,z}  \delta{p}_{i,x}  \delta{p}_{i,y}  \delta{p}_{i,z}  \delta{v ↪

↪ }_{i,x}  \delta{v}_{i,y}  \delta{v}_{i,z}  \delta{\theta}_{i,x}  \delta{\the ↪

↪ ta}_{i,y}  \delta{\theta}_{i,z}]

---
# Measurements

## IMU

$$
h_g = \mathbf{R}^{\{q\}}g + a_b
$$

In [16]:
h_g = R(q).T * sym.Matrix([0, 0, gravity]) + a_b
h_g

⎡a_{b,x} + g⋅(-2⋅q_w⋅q_y + 2⋅qₓ⋅q_z)⎤
⎢                                   ⎥
⎢a_{b,y} + g⋅(2⋅q_w⋅qₓ + 2⋅q_y⋅q_z) ⎥
⎢                                   ⎥
⎢            ⎛      2        2    ⎞ ⎥
⎣a_{b,z} + g⋅⎝- 2⋅qₓ  - 2⋅q_y  + 1⎠ ⎦

## Face detection

\begin{align*}
h_{sq} &= (q \otimes \Delta q_{I \to c})^{-1} \otimes q_i \\
h_{sp} &= \mathbf{R}^{\{\Delta q_{I \to c}\}}(\mathbf{R}^{\{q\}}(p_i - p) - \Delta p_{I \to c}) \\
\\
\Delta p_{I \to c} &= \begin{bmatrix}
0 & -0.0013 & - 0.00662
\end{bmatrix}^\top \\
\Delta q_{I \to c} &= \begin{bmatrix}
\displaystyle \cos\left(-\frac{102°}{2}\right) & \displaystyle \sin\left(-\frac{102°}{2}\right) & 0 & 0
\end{bmatrix}^\top \\
\end{align*}

In [17]:
h_sp = R(IMU_to_cam_rotation) * (R(q) * (p_i - p) - IMU_to_cam_translation)
h_sq = left_quat_matrix(quat_inv(left_quat_matrix(q) * IMU_to_cam_rotation)) * q_i

h_sp

⎡                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                          ⎛                                                   ↪
⎢- 2⋅c_{\phi/2}⋅s_{\phi/2}⋅⎝-\Delta{p}_z + (-pₓ + p_{i,x})⋅(-2⋅q_w⋅q_y + 2⋅qₓ⋅ ↪
⎢                                                                              ↪
⎢                         ⎛                                                    ↪
⎣ 2⋅c_{\phi/2}⋅s_{\phi/2}⋅⎝-\Delta{p}_y + (-pₓ + p_{i,x})⋅(2⋅q_w⋅q_z + 2⋅qₓ⋅q_ ↪

↪                                                 ⎛       2        2    ⎞      ↪
↪                   -\Delta{p}ₓ + (-pₓ + p_{i,x})⋅⎝- 2⋅q_y  - 2⋅q_z  + 1⎠ + (- ↪
↪                                                                              ↪
↪                                                                   ⎛      2   ↪
↪ q_z) + (-p_y + p_{i,y})⋅(

## Combined Measurements

\begin{align*}
h_g &= \mathbf{R}^{\{q\}}g + a_b \\
h_{sq} &= (q \otimes \Delta q_{I \to c})^{-1} \otimes q_i \\
h_{sp} &= \mathbf{R}^{\{\Delta q_{I \to c}\}}(\mathbf{R}^{\{q\}}(p_i - p) - \Delta p_{I \to c}) \\
\end{align*}

In [18]:
h = sym.Matrix.vstack(h_g, h_sp, h_sq)

h

⎡                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                          ⎛                                                   ↪
⎢- 2⋅c_{\phi/2}⋅s_{\phi/2}⋅⎝-\Delta{p}_z + (-pₓ + p_{i,x})⋅(-2⋅q_w⋅q_y + 2⋅qₓ⋅ ↪
⎢                           

---
# Jacobians for the Filter Equations

## State Transition Jacobian $F_{\delta x}$

In [37]:
# TODO add all the speaker stuff
F_delta_x = f_error_state.jacobian(delta_x)

F_delta_x_n = f_error_state.jacobian(delta_x_n)
# since we always just add the noise, the following line would yield the same noise jacobian
# sym.eye(24)[:,3:]

F_delta_x_n

⎡0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0⎤
⎢                                                             ⎥
⎢0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0⎥
⎢                                                             ⎥
⎢0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0⎥
⎢                                                             ⎥
⎢1  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0⎥
⎢                                                             ⎥
⎢0  1  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0⎥
⎢                                                             ⎥
⎢0  0  1  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0⎥
⎢                                                             ⎥
⎢0  0  0  1  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0⎥
⎢                                                             ⎥
⎢0  0  0  0  1  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0⎥
⎢                                       

## Measurement Jacobian $H$

$$
H = \frac{\partial h}{\partial \delta x} = \frac{\partial h}{\partial x_t} \frac{\partial x_t}{\partial \delta x} = H_x X_{\delta x}
$$
The true state is just the nominal state with the error injected $x_t = x \oplus \delta x$ (special case for quaternions)

In [38]:
#true state
p_v_true = x_IMU[0:6, :] + delta_x_IMU[0:6, :]
q_true = left_quat_matrix(x_IMU[6:10, :]) * quat_from_axis_angle(delta_x_IMU[6:9, :])
a_b_omega_b_true = x_IMU[10:16, :] + delta_x_IMU[9:15, :]

p_i_v_i_true = x_speaker[0:6, :] + delta_x_speaker[0:6, :]
q_i_true = left_quat_matrix(x_speaker[6:10, :]) * quat_from_axis_angle(delta_x_speaker[6:9, :])

true_state = sym.Matrix.vstack(p_v_true, q_true, a_b_omega_b_true, p_i_v_i_true, q_i_true)

H_x = h.jacobian(x)
X_delta_x = true_state.jacobian(delta_x)

H_delta_x = sym.simplify(H_x * X_delta_x)

H_delta_x
X_delta_x

⎡1  0  0  0  0  0     0         0         0      0  0  0  0  0  0  0  0  0  0  ↪
⎢                                                                              ↪
⎢0  1  0  0  0  0     0         0         0      0  0  0  0  0  0  0  0  0  0  ↪
⎢                                                                              ↪
⎢0  0  1  0  0  0     0         0         0      0  0  0  0  0  0  0  0  0  0  ↪
⎢                                                                              ↪
⎢0  0  0  1  0  0     0         0         0      0  0  0  0  0  0  0  0  0  0  ↪
⎢                                                                              ↪
⎢0  0  0  0  1  0     0         0         0      0  0  0  0  0  0  0  0  0  0  ↪
⎢                                                                              ↪
⎢0  0  0  0  0  1     0         0         0      0  0  0  0  0  0  0  0  0  0  ↪
⎢                                                                              ↪
⎢0  0  0  0  0  0  -0.5⋅qₓ  

---
# Observability

## Weak Observability of Nominal State

\begin{align*}
\text{rank}\left( \begin{bmatrix}
\frac{\partial L_f^0 h(x)}{\partial x} \\
\frac{\partial L_f^1 h(x)}{\partial x} \\
\vdots \\
\frac{\partial L_f^{n-1} h(x)}{\partial x} \\
\end{bmatrix} \right) \stackrel{?}{=} n
\end{align*}

In [ ]:
%%script true
# skipping the cell for when I rerun whole notebook

# Setup your symbols
dt = sym.Symbol('\\Delta t')
gravity = sym.Symbol('g')
IMU_to_cam_translation = sym.Matrix(sym.symbols("\\Delta{p}_x, \\Delta{p}_y, \\Delta{p}_z"))
angle = sym.Symbol("\\phi")
IMU_to_cam_rotation = sym.Matrix([sym.cos(-angle / 2), sym.sin(-angle / 2), 0, 0])

n_states = x.shape[0]  # 26 states

# ==============================================================================
# 1. TRIGONOMETRIC SUBSTITUTION (Using your `angle` variable)
# ==============================================================================
s_phi_2, c_phi_2 = sym.symbols('s_phi_2 c_phi_2', real=True)

# SymPy converts sin(-angle/2) to -sin(angle/2) automatically,
# so replacing sin(angle/2) and cos(angle/2) covers both positive and negative cases.
trig_replacements = {
    sym.sin(angle / 2): s_phi_2,
    sym.cos(angle / 2): c_phi_2,
}

# Apply to raw models FIRST before taking Jacobians
h_sub = h.subs(trig_replacements)
f_sub = f_nominal_state.subs(trig_replacements)

# ==============================================================================
# 2. BUILD SYMBOLIC OBSERVABILITY MATRIX
# ==============================================================================
print("Building symbolic observability matrix...")
H_prev = h_sub.jacobian(x)
observability_matrix = H_prev

for k in range(1, 3):
    Lf_k = H_prev * f_sub
    H_k = Lf_k.jacobian(x)
    observability_matrix = sym.Matrix.vstack(observability_matrix, H_k)
    H_prev = H_k
    print(f"Order {k}/{n_states - 1} appended. Matrix shape: {observability_matrix.shape}")

# ==============================================================================
# 3. NUMERICAL RANK EVALUATION (SVD via NumPy)
# ==============================================================================
print("\nEvaluating numerical rank...")

# Automatically grab all remaining free symbols (including s_phi_2 and c_phi_2)
free_symbols = list(observability_matrix.free_symbols)

# Lambdify symbolic matrix to a vectorized numpy function
obs_func = sym.lambdify([free_symbols], observability_matrix, modules="numpy")

# Generate non-zero random numerical inputs
np.random.seed(42)
random_vals = np.random.randn(len(free_symbols))

# Substitute values and compute rank
O_num = obs_func(random_vals)
rank = np.linalg.matrix_rank(O_num)

# ==============================================================================
# 4. RESULTS
# ==============================================================================
print(f"Final Observability Matrix Shape: {O_num.shape}")
print(f"System Numerical Rank: {rank} / {n_states}")

if rank == n_states:
    print("System Status: FULLY OBSERVABLE")
else:
    print(f"System Status: UN-OBSERVABLE ({n_states - rank} unobservable states)")

## Weak Observability of Error State

Since the above calculation takes a long time, I will use the second option of showing observability.

$$\mathcal{O} = \begin{bmatrix} H \\ HF \\ HF^2 \\ \vdots \\ HF^{n-1} \end{bmatrix}$$

With

\begin{align*}
H &= \frac{\partial h(x \oplus \delta x)}{\partial (x \oplus \delta x)} \cdot \frac{\partial (x \oplus \delta x)}{\partial \delta x} = \frac{\partial h}{\partial x_t} \frac{\partial x_t}{\partial \delta x} \\
F &= \frac{\partial f(x \oplus \delta x)}{\partial (x \oplus \delta x)} \cdot \frac{\partial (x \oplus \delta x)}{\partial \delta x} = \frac{\partial f}{\partial x_t} \frac{\partial x_t}{\partial \delta x}
\end{align*}

In [ ]:
n_states = delta_x.shape[0]
print(f"number of variables in state {n_states}")

#true state
p_v_true = x_IMU[0:6, :] + delta_x_IMU[0:6, :]
q_true = left_quat_matrix(x_IMU[6:10, :]) * quat_from_axis_angle(delta_x_IMU[6:9, :])
a_b_omega_b_true = x_IMU[10:16, :] + delta_x_IMU[9:15, :]

p_i_v_i_true = x_speaker[0:6, :] + delta_x_speaker[0:6, :]
q_i_true = left_quat_matrix(x_speaker[6:10, :]) * quat_from_axis_angle(delta_x_speaker[6:9, :])

true_state = sym.Matrix.vstack(p_v_true, q_true, a_b_omega_b_true, p_i_v_i_true, q_i_true)

print(f"true state constructed")

H = h.jacobian(x) * true_state.jacobian(delta_x)
print(f"H matrix constructed")
F = f_error_state.jacobian(delta_x)
# F = f_nominal_state.jacobian(x) * true_state.jacobian(delta_x)
print(f"F matrix constructed")

# Create a dictionary mapping all delta_x symbols to 0
zero_delta = {var: 0 for var in delta_x}

# Substitute zero error
H = H.subs(zero_delta)
F = F.subs(zero_delta)

H = sym.simplify(H)
print(f"H matrix simplified")
F = sym.simplify(F)
print(f"F matrix simplified")

H.shape, F.shape

number of variables in state 24
true state constructed
H matrix constructed
F matrix constructed
H matrix simplified
F matrix simplified


In [ ]:
blocks = [H]
HF_k = H

for k in range(1, 6):
    HF_k = HF_k * F  # Pure matrix multiplication (fast!)
    blocks.append(HF_k)
    print(f"k = {k} calculated")

observability_matrix = sym.Matrix.vstack(*blocks)
observability_matrix.shape

k = 1 calculated
k = 2 calculated
k = 3 calculated
k = 4 calculated
k = 5 calculated


In [ ]:
# ==============================================================================
# 3. NUMERICAL RANK EVALUATION (SVD via NumPy)
# ==============================================================================
print("\nEvaluating numerical rank...")

# Automatically grab all remaining free symbols (including s_phi_2 and c_phi_2)
free_symbols = list(observability_matrix.free_symbols)

# Lambdify symbolic matrix to a vectorized numpy function
obs_func = sym.lambdify([free_symbols], observability_matrix, modules="numpy")

# Generate non-zero random numerical inputs
np.random.seed(42)
random_vals = np.random.randn(len(free_symbols))

# Substitute values and compute rank
O_num = obs_func(random_vals)
rank = np.linalg.matrix_rank(O_num)

# ==============================================================================
# 4. RESULTS
# ==============================================================================
print(f"Final Observability Matrix Shape: {O_num.shape}")
print(f"System Numerical Rank: {rank} / {n_states}")

if rank == n_states:
    print("System Status: FULLY OBSERVABLE")
else:
    print(f"System Status: UN-OBSERVABLE ({n_states - rank} unobservable states)")


Evaluating numerical rank...
Final Observability Matrix Shape: (60, 24)
System Numerical Rank: 17 / 24
System Status: UN-OBSERVABLE (7 unobservable states)


In [ ]:
# ==============================================================================
# HYPOTHESIS TESTING FOR ALL 7 UNOBSERVABLE MODES
# ==============================================================================


def test_hypothesis(v_test, description):
    residual = np.linalg.norm(O_num @ v_test)
    status = "UNOBSERVABLE (In Nullspace)" if residual < 1e-4 else "OBSERVABLE"
    print(f"{description:<35} | Residual: {residual:.6e} | {status}")


print("\n--- HYPOTHESIS TESTING RESULTS ---")

# 1-3. Global Position Translations (X, Y, Z)
for axis_idx, axis_name in enumerate(["X", "Y", "Z"]):
    v = np.zeros(24)
    v[0 + axis_idx] = 1.0  # IMU position shift
    v[15 + axis_idx] = 1.0  # Speaker position shift
    test_hypothesis(v, f"Global Position {axis_name} Translation")

# 4-6. Global Velocity Translations (Vx, Vy, Vz)
for axis_idx, axis_name in enumerate(["X", "Y", "Z"]):
    v = np.zeros(24)
    v[3 + axis_idx] = 1.0  # IMU velocity shift
    v[18 + axis_idx] = 1.0  # Speaker velocity shift
    test_hypothesis(v, f"Global Velocity {axis_name} Translation")

# 7. Global Yaw Rotation (around Z axis)
# An infinitesimal yaw rotation delta_psi around global Z transforms position and velocity:
# delta_p = [-p_y, p_x, 0]^T and delta_v = [-v_y, v_x, 0]^T
# Assumes position/velocity values from your numerical evaluation point:
p_x, p_y = random_vals[0], random_vals[1]  # IMU position at eval point
v_x, v_y = random_vals[3], random_vals[4]  # IMU velocity at eval point
pi_x, pi_y = random_vals[15], random_vals[16]  # Speaker position at eval point
vi_x, vi_y = random_vals[18], random_vals[19]  # Speaker velocity at eval point

v_yaw = np.zeros(24)
# IMU state rotation effect
v_yaw[0], v_yaw[1] = -p_y, p_x
v_yaw[3], v_yaw[4] = -v_y, v_x
v_yaw[8] = 1.0  # IMU delta_th_z (yaw error)

# Speaker state rotation effect
v_yaw[15], v_yaw[16] = -pi_y, pi_x
v_yaw[18], v_yaw[19] = -vi_y, vi_x
v_yaw[23] = 1.0  # Speaker delta_thi_z (yaw error)

test_hypothesis(v_yaw, "Global Yaw Rotation (Z-Axis)")


--- HYPOTHESIS TESTING RESULTS ---
Global Position X Translation       | Residual: 0.000000e+00 | UNOBSERVABLE (In Nullspace)
Global Position Y Translation       | Residual: 0.000000e+00 | UNOBSERVABLE (In Nullspace)
Global Position Z Translation       | Residual: 0.000000e+00 | UNOBSERVABLE (In Nullspace)
Global Velocity X Translation       | Residual: 0.000000e+00 | UNOBSERVABLE (In Nullspace)
Global Velocity Y Translation       | Residual: 0.000000e+00 | UNOBSERVABLE (In Nullspace)
Global Velocity Z Translation       | Residual: 0.000000e+00 | UNOBSERVABLE (In Nullspace)
Global Yaw Rotation (Z-Axis)        | Residual: 1.586972e+03 | OBSERVABLE


In [ ]:
# ==============================================================================
# HYPOTHESIS TESTING FOR RELATIVE POSITION (OBSERVABLE)
# ==============================================================================
print("\n--- RELATIVE POSITION HYPOTHESIS TEST ---")

for axis_idx, axis_name in enumerate(["X", "Y", "Z"]):
    v_relative = np.zeros(24)

    # Keep IMU fixed (v_relative[0 + axis_idx] = 0.0)
    # Shift Speaker position by +1.0 unit
    v_relative[15 + axis_idx] = 1.0

    residual = np.linalg.norm(O_num @ v_relative)
    status = (
        "OBSERVABLE (Not in Nullspace)" if residual > 1e-4 else "UNOBSERVABLE"
    )

    print(
        f"Relative Position {axis_name} Shift | Residual: {residual:.6e} |"
        f" {status}"
    )



--- RELATIVE POSITION HYPOTHESIS TEST ---
Relative Position X Shift | Residual: 6.466270e+01 | OBSERVABLE (Not in Nullspace)
Relative Position Y Shift | Residual: 8.001989e+01 | OBSERVABLE (Not in Nullspace)
Relative Position Z Shift | Residual: 1.197425e+02 | OBSERVABLE (Not in Nullspace)


In [ ]:
%%script true
#true state
p_v_true = x_IMU[0:6, :] + delta_x_IMU[0:6, :]
q_true = left_quat_matrix(x_IMU[6:10, :]) * quat_from_axis_angle(delta_x_IMU[6:9, :])
a_b_omega_b_true = x_IMU[10:16, :] + delta_x_IMU[9:15, :]

p_i_v_i_true = x_speaker[0:6, :] + delta_x_speaker[0:6, :]
q_i_true = left_quat_matrix(x_speaker[6:10, :]) * quat_from_axis_angle(delta_x_speaker[6:9, :])

true_state = sym.Matrix.vstack(p_v_true, q_true, a_b_omega_b_true, p_i_v_i_true, q_i_true)

H = h.jacobian(x) * true_state.jacobian(delta_x)

sym.simplify(H)

In [ ]:
%%script true
# 4. Fast Numerical Rank via NumPy
print("Evaluating numerical rank...")
free_symbols = list(observability_matrix.free_symbols)
obs_func = sym.lambdify([free_symbols], observability_matrix, modules="numpy")

np.random.seed(42)
random_vals = np.random.randn(len(free_symbols))

O_num = obs_func(random_vals)
rank = np.linalg.matrix_rank(O_num)

print(f"\nFinal Observability Matrix Shape: {O_num.shape}")
print(f"System Numerical Rank: {rank} / {n_states}")
print(f"Unobservable States: {n_states - rank}")

---
# debugging

In [39]:
# quaternion test
q_test = sym.Matrix(sym.symbols("q_w, q_x, q_y, q_z"))
p = sym.Matrix(sym.symbols("x, y, z"))

# axis angle test
result_b = quat_from_axis_angle(p, False)

# skew-symmetric matrix test
skew_symmetric_matrix(p)

# H matrix only for IMU
true_state_IMU = sym.Matrix.vstack(p_v_true, q_true, a_b_omega_b_true)

H_x_IMU = h_g.jacobian(x_IMU)
X_delta_x_IMU = true_state_IMU.jacobian(delta_x_IMU)

H_delta_x_IMU = H_x_IMU * X_delta_x_IMU
sym.simplify(H_delta_x_IMU)

⎡                                                          ⎛     2     2       ↪
⎢0  0  0  0  0  0                 0                  1.0⋅g⋅⎝- q_w  + qₓ  + q_y ↪
⎢                                                                              ↪
⎢                        ⎛   2     2      2      2⎞                            ↪
⎢0  0  0  0  0  0  1.0⋅g⋅⎝q_w  - qₓ  - q_y  + q_z ⎠                  0         ↪
⎢                                                                              ↪
⎣0  0  0  0  0  0     2.0⋅g⋅(-q_w⋅qₓ - q_y⋅q_z)          2.0⋅g⋅(-q_w⋅q_y + qₓ⋅ ↪

↪ 2      2⎞                                            ⎤
↪   - q_z ⎠  2.0⋅g⋅(q_w⋅qₓ + q_y⋅q_z)  1  0  0  0  0  0⎥
↪                                                      ⎥
↪                                                      ⎥
↪            2.0⋅g⋅(q_w⋅q_y - qₓ⋅q_z)  0  1  0  0  0  0⎥
↪                                                      ⎥
↪ q_z)                  0              0  0  1  0  0  0⎦